<a href="https://colab.research.google.com/github/Yami2912/HER360/blob/feature%2Fhealthcare/06_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os, pathlib, sys, json
TOKEN = userdata.get("GITHUB_TOKEN")
os.chdir("/content")
if not os.path.exists("/content/HER360"):
    !git clone -b feature/healthcare https://{TOKEN}@github.com/Yami2912/HER360.git
%cd /content/HER360
!git config user.name  "Yami2912"
!git config user.email "yamipatel0618@gmail.com"
!git fetch origin
!git checkout feature/healthcare
!git pull origin feature/healthcare

BASE = pathlib.Path("/content/HER360/HER360-Healthcare-ML")
print(BASE.exists())

Cloning into 'HER360'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (207/207), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 207 (delta 37), reused 89 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (207/207), 1.63 MiB | 7.44 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/HER360
Already on 'feature/healthcare'
Your branch is up to date with 'origin/feature/healthcare'.
From https://github.com/Yami2912/HER360
 * branch            feature/healthcare -> FETCH_HEAD
Already up to date.
True


In [2]:
import numpy as np, pandas as pd, warnings, datetime
warnings.filterwarnings("ignore")
from sklearn.model_selection import GroupKFold, cross_val_predict

sys.path.insert(0, str(BASE / "preprocessing/fertility"))
sys.path.insert(0, str(BASE / "models/fertility/ovulation_predictor"))
import ovulation_prep as op
import ovulation_estimators as oe

T, G, F = op.TARGET, op.GROUP, op.FEATURES_V1
NB_DIR    = BASE / "notebooks/fertility/ovulation_predictor"
MODEL_DIR = BASE / "models/fertility/ovulation_predictor"

# 1. The decision made in notebook 05
decision = json.loads((NB_DIR / "model_decision_v1.json").read_text())
print("Decision file says:", decision["chosen_v1_approach"])
assert decision["chosen_v1_approach"] == "calendar_offset", "Decision changed - tell Claude before continuing"

# 2. Split integrity
df = pd.read_csv(BASE / "datasets/fertility/ovulation_predictor/processed/ovulation_model_table_v1.csv")
parts = {s: set(df[df.split == s][G]) for s in ["train", "validation", "test"]}
assert not (parts["train"] & parts["validation"]) and not (parts["train"] & parts["test"]) \
       and not (parts["validation"] & parts["test"]), "participant appears in two splits"
print("rows per split:", df.split.value_counts().to_dict())

# 3. Helpers
def hist(d):
    """Rows with at least one previous cycle (the only rows we give an estimate for)."""
    return d[d.n_prev_cycles >= 1].reset_index(drop=True)

def metrics(y, p):
    e = np.abs(y - p)
    return {"n": int(len(y)),
            "MAE": round(float(e.mean()), 2),
            "RMSE": round(float(np.sqrt(((y - p) ** 2).mean())), 2),
            "within_1_day_%": round(100 * float((e <= 1).mean()), 1),
            "within_2_days_%": round(100 * float((e <= 2).mean()), 1)}

def fit_final(fit_df):
    """Fit the calendar offset, and the +/- range from out-of-fold errors. Never sees test rows."""
    h = hist(fit_df)
    model = oe.CalendarOffset().fit(h[F], h[T])
    oof = cross_val_predict(oe.CalendarOffset(), h[F], h[T],
                            cv=GroupKFold(n_splits=5), groups=h[G])
    half = float(np.quantile(np.abs(h[T] - oof), 0.80))      # range aims to contain the true day ~80% of the time
    return model, half, h

def score(model, half, eval_df):
    ev = hist(eval_df)
    p = model.predict(ev[F])
    inside = lambda y, q: np.abs(y - q) <= half
    overall = {**metrics(ev[T], p), "interval_coverage_%": round(100 * float(inside(ev[T], p).mean()), 1)}
    by_status = {s: {**metrics(g[T], p[g.index]),
                     "interval_coverage_%": round(100 * float(inside(g[T], p[g.index]).mean()), 1)}
                 for s, g in ev.groupby("history_status")}
    return overall, by_status, ev
print("ready")

Decision file says: calendar_offset
rows per split: {'train': 1070, 'test': 228, 'validation': 204}
ready


In [3]:
LOCK = NB_DIR / "test_evaluation_v1.json"

if LOCK.exists():
    print("TEST SPLIT ALREADY SCORED - showing the stored result, not recomputing.\n")
    print(json.dumps(json.loads(LOCK.read_text()), indent=2))
else:
    fit_df  = df[df.split.isin(["train", "validation"])]    # final fit uses train + validation
    test_df = df[df.split == "test"]

    model, half, h = fit_final(fit_df)
    overall, by_status, ev = score(model, half, test_df)

    reference = metrics(ev[T], oe.GlobalMedian().fit(h[F], h[T]).predict(ev[F]))   # context only

    result = {
        "evaluated_on": "test",
        "approach": "calendar_offset",
        "offset_days": round(model.offset_, 4),
        "interval_half_width_days": round(half, 2),
        "rows_total": int(len(test_df)),
        "rows_without_estimate_insufficient_history": int((test_df.n_prev_cycles == 0).sum()),
        "participants_scored": int(ev[G].nunique()),
        "overall_rows_with_history": overall,
        "by_history_status": by_status,
        "reference_global_median": reference,
        "note": "Scored once. Not medical guidance.",
    }
    LOCK.write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))

{
  "evaluated_on": "test",
  "approach": "calendar_offset",
  "offset_days": -13.3333,
  "interval_half_width_days": 3.67,
  "rows_total": 228,
  "rows_without_estimate_insufficient_history": 24,
  "participants_scored": 22,
  "overall_rows_with_history": {
    "n": 204,
    "MAE": 2.01,
    "RMSE": 2.79,
    "within_1_day_%": 39.2,
    "within_2_days_%": 64.2,
    "interval_coverage_%": 86.8
  },
  "by_history_status": {
    "limited": {
      "n": 60,
      "MAE": 1.98,
      "RMSE": 2.91,
      "within_1_day_%": 43.3,
      "within_2_days_%": 63.3,
      "interval_coverage_%": 86.7
    },
    "sufficient": {
      "n": 144,
      "MAE": 2.02,
      "RMSE": 2.75,
      "within_1_day_%": 37.5,
      "within_2_days_%": 64.6,
      "interval_coverage_%": 86.8
    }
  },
  "reference_global_median": {
    "n": 204,
    "MAE": 2.28,
    "RMSE": 3.21,
    "within_1_day_%": 46.6,
    "within_2_days_%": 68.1
  },
  "note": "Scored once. Not medical guidance."
}


In [4]:
model, half, h = fit_final(df[df.split.isin(["train", "validation"])])
test_result = json.loads(LOCK.read_text())

artefact = {
    "model_version": "ovulation-calendar-v1",
    "approach": "calendar_offset",
    "formula": "estimated_ovulation_day = mean(previous cycle lengths, up to 3) + offset_days",
    "offset_days": round(model.offset_, 4),
    "interval_half_width_days": round(half, 2),
    "interval_nominal_coverage": 0.8,
    "min_previous_cycles": 1,
    "history_status_rule": {"insufficient": "0 previous cycles (no estimate)",
                            "limited": "1-2 previous cycles",
                            "sufficient": "3 previous cycles"},
    "features": F,
    "trained_on": {"splits": ["train", "validation"], "rows": int(len(h)),
                   "participants": int(h[G].nunique())},
    "test_metrics": test_result["overall_rows_with_history"],
    "created": datetime.date.today().isoformat(),
    "limitations": [
        "Target is a source-estimated ovulation day, not confirmed ovulation",
        "About 157 participants; cannot validate irregular or PCOS cycles",
        "Estimate only - not proof of ovulation and not contraception",
        "College project dataset; replace with professionally verified data before real use",
    ],
}
(MODEL_DIR / "ovulation_calendar_v1.json").write_text(json.dumps(artefact, indent=2))
print(json.dumps(artefact, indent=2))

{
  "model_version": "ovulation-calendar-v1",
  "approach": "calendar_offset",
  "formula": "estimated_ovulation_day = mean(previous cycle lengths, up to 3) + offset_days",
  "offset_days": -13.3333,
  "interval_half_width_days": 3.67,
  "interval_nominal_coverage": 0.8,
  "min_previous_cycles": 1,
  "history_status_rule": {
    "insufficient": "0 previous cycles (no estimate)",
    "limited": "1-2 previous cycles",
    "sufficient": "3 previous cycles"
  },
  "features": [
    "prev_cycle_length_1",
    "prev_cycle_length_2",
    "prev_cycle_length_3",
    "prev_cycle_mean",
    "prev_cycle_std",
    "n_prev_cycles"
  ],
  "trained_on": {
    "splits": [
      "train",
      "validation"
    ],
    "rows": 1137,
    "participants": 121
  },
  "test_metrics": {
    "n": 204,
    "MAE": 2.01,
    "RMSE": 2.79,
    "within_1_day_%": 39.2,
    "within_2_days_%": 64.2,
    "interval_coverage_%": 86.8
  },
  "created": "2026-11-26",
  "limitations": [
    "Target is a source-estimated ovula

In [5]:
%cd /content/HER360
!git add HER360-Healthcare-ML/models HER360-Healthcare-ML/notebooks
!git status --short

/content/HER360
A  HER360-Healthcare-ML/models/fertility/ovulation_predictor/ovulation_calendar_v1.json
A  HER360-Healthcare-ML/notebooks/fertility/ovulation_predictor/test_evaluation_v1.json
